# FASAN — Fältbaserad AI för Sökning Av Nödställda

**Identifiera människor i terräng — oavsett om de gömmer sig eller inte. Uppdrag åt Försvarsmakten.**

Den här notebooken går igenom hela vårt arbetsflöde: från rådata till en tränad
modell som avgör om det finns en människa i en bild eller inte. Vi *importerar*
projektets moduler i stället för att klistra in koden på nytt, så att notebooken
speglar exakt samma kod som körs i pipelinen (`pipeline.py`).

> **Om körning:** Notebooken är gjord för att *visa flödet och de sparade resultaten*.
> De lätta stegen (datakontroll, dataladdning, modellbygge) kan köras direkt. De tunga
> stegen — träning och Grad-CAM över hela testmängden — körs **inte** om här; i stället
> laddar vi den sparade modellen och visar de färdiga graferna och rapporterna från
> `output/`. Kör cellerna uppifrån och ned i miljön `Tensorflow_AI` från projektroten.

### Datasetet
[SARD – Search and Rescue](https://www.kaggle.com/datasets/nikolasgegenava/sard-search-and-rescue)
innehåller flygbilder över terräng med YOLO-etiketter. Vi gör om uppgiften till
**binär klassificering**: en bild med minst en YOLO-box får etikett `1` (människa),
en bild med tom etikettfil får `0` (ingen människa).


## 1. Setup och imports

All konfiguration ligger samlad i `config.py` (bildstorlek, batch-storlek,
träningsparametrar, sökvägar m.m.). Funktionaliteten är uppdelad i moduler som vi
importerar nedan:

| Modul | Ansvar |
|---|---|
| `clean_data` | Kontrollerar/rensar bilder utan etikett och tvärtom |
| `load_data` | Läser bilder + YOLO-etiketter, batchar, beräknar class weights |
| `eda` / `graphs` | EDA-grafer och utvärderingsgrafer |
| `model` | Bygger CNN-arkitekturen |
| `training` | Tränings-loop, callbacks, prediktion, rapporter |
| `grad_cam` | Grad-CAM-värmekartor och IoU-utvärdering |


In [ ]:
import config

print("Bildstorlek (input):", config.IMAGE_SIZE)
print("Batch-storlek:      ", config.BATCH_SIZE)
print("Klasser:            ", config.CLASS_NAMES)
print("Epoker (max):       ", config.EPOCHS)
print("Learning rate:      ", config.LEARNING_RATE)
print("Class weight-läge:  ", config.CLASS_WEIGHT_MODE)


## 2. Datakontroll och rensning

Innan något annat kontrolleras datasetet: finns det bilder utan matchande
etikettfil, eller etikettfiler utan bild? I pipelinen körs detta som *rapportering*
(`APPLY_DATA_CLEANING = False`) — problem listas men inga filer flyttas. Sätts flaggan
till `True` flyttas problemfiler i stället till `dataset/_removed`.


In [ ]:
from clean_data import clean_data, print_clean_report

clean_report = clean_data(config.DATASET_PATH, apply_changes=config.APPLY_DATA_CLEANING)
print_clean_report(clean_report)


## 3. Ladda data och klassfördelning

`load_data` returnerar `train`, `valid` och `test` som `SplitData`-objekt. Bildernas
sökvägar samlas in direkt, men själva pixlarna laddas först när en batch behövs — det
håller minnesanvändningen nere trots tusentals bilder.

YOLO-etiketterna översätts till binära klasser i `read_binary_label`: tom fil → `0`,
fil med minst en giltig YOLO-rad → `1`.


In [ ]:
from load_data import load_data, calculate_class_weights, print_dataset_summary

train_data, valid_data, test_data = load_data(config.DATASET_PATH)
print_dataset_summary(train_data, valid_data, test_data)


Datasetet är **obalanserat** — det finns betydligt fler bilder med människa än utan.
För att den mindre klassen ändå ska få genomslag under träningen beräknar vi
*class weights* (läge `balanced`).


In [ ]:
class_weights = calculate_class_weights(train_data)
print("Class weights:", class_weights)


Klassfördelningen per datasetdel (sparad EDA-graf):

![Klassfördelning per datasetdel](output/eda/class_distribution.png)

Obalansen är tydlig och motiverar både class weights och att vi senare tittar extra
noga på *recall* för klassen `human`.


## 4. Utforskande dataanalys (EDA)

EDA-graferna skapas av `eda.create_eda_graphs` innan träningen, så att de finns
dokumenterade även om gamla output-bilder rensats. Nedan visas de sparade graferna.

**Datareduktion.** Originalbilderna skalas ned till modellens input-storlek
(640×640). Grafen visar hur mycket pixeldata som reduceras per bild:

![Datareduktion genom bildstorlek](output/eda/image_size_reduction.png)

**Exempelbilder.** En snabb visuell kontroll av att bilder och etiketter ser rimliga
ut. Människorna är ofta små och kan vara delvis dolda i terrängen — vilket är hela
utmaningen i Search and Rescue.

![Exempelbilder, train](output/eda/sample_images_train.png)


Vill man återskapa EDA-graferna från den laddade datan kan man köra cellen nedan.
Den skriver om bilderna till `output/eda/`.


In [ ]:
# from eda import create_eda_graphs
# graph_paths = create_eda_graphs(
#     [train_data, valid_data, test_data],
#     output_dir=config.EDA_OUTPUT_DIR,
#     sample_count=config.EDA_SAMPLE_COUNT,
# )
# graph_paths


## 5. CNN-modellen

Arkitekturen ligger i `model.build_cnn_model` och styrs helt från `config.py`. Det är
ett klassiskt CNN för binär klassificering:

- Fyra convolution-block (`[32, 64, 128, 256]` filter), vardera med två `Conv2D`-lager
  följt av `MaxPooling2D`.
- `GlobalMaxPooling2D` sammanfattar feature-maps utan ett stort flatten-lager.
- Två `Dense`-lager (256 units) med L2-regularisering och dropout.
- Ett `Dense(1, sigmoid)` som ger sannolikheten för "människa".

Optimerare är `AdamW` och loss är `binary_crossentropy`.


In [ ]:
from model import build_cnn_model

model = build_cnn_model()
model.summary()


## 6. Träning

Träningen sköts av `training.train_cnn_model`. Träningsdatan läses via en
`ImageSequence` (en Keras `Sequence`) som blandar och **augmenterar** bilderna mellan
epokerna, medan valid/test alltid lämnas oförändrade. Augmenteringen är medvetet mild
(flip, lätt rotation, ljus/kontrast/färg) så att bilderna fortfarande ser realistiska ut.

Callbacks håller träningen i schack:
- **EarlyStopping** (`val_loss`, patience 3) återställer de bästa vikterna.
- **ReduceLROnPlateau** sänker learning rate när valideringsförlusten planar ut.

> Vi **tränar inte om** här. Koden nedan visas för fullständighetens skull (utkommenterad).
> I stället laddar vi den färdigtränade modellen i nästa cell.


In [ ]:
# from training import train_cnn_model, create_training_output_dirs, print_best_epoch_summary
#
# create_training_output_dirs()
# history = train_cnn_model(model, train_data, valid_data, class_weights)
# print_best_epoch_summary(history)


In [ ]:
import tensorflow as tf

model_path = config.CHECKPOINT_OUTPUT_DIR / config.MODEL_FILE_NAME
trained_model = tf.keras.models.load_model(model_path)
print("Laddade sparad modell:", model_path)


Träningshistoriken från den sparade körningen — loss och accuracy per epoch för
både träning och validering:

![Träningshistorik](output/training/training_history.png)


## 7. Utvärdering på testdata

Testmängden (570 bilder modellen aldrig sett) ger en rättvis bild. Modellen ger
sannolikheter via sigmoid, som tröskas vid `0.5` till klasser. Vi visar den sparade
`classification_report` samt confusion matrix. Cellen under läser rapportfilen direkt;
vill man räkna om från modellen finns koden utkommenterad.


In [ ]:
report_path = config.TRAINING_OUTPUT_DIR / config.CLASSIFICATION_REPORT_FILE_NAME
print(report_path.read_text(encoding="utf-8"))


In [ ]:
# Räkna om classification report från den laddade modellen (tar en stund):
# from training import predict_probabilities, predict_classes, create_classification_report
# probabilities = predict_probabilities(trained_model, test_data)
# predictions = predict_classes(probabilities)
# print(create_classification_report(test_data.labels, predictions))


**Tolkning.** Modellen hittar de flesta människorna: *recall* 0.91 och *precision*
0.97 för `human`. Priset är fler falsklarm på `no human` (precision 0.61) — men i
Search and Rescue är det betydligt värre att *missa* en människa än att få ett
falsklarm, så avvägningen är rimlig.

![Confusion matrix](output/training/confusion_matrix.png)


## 8. Grad-CAM och IoU — *var* tittar modellen?

En hög träffsäkerhet räcker inte: vi vill veta om modellen tittar på rätt saker.
`grad_cam` beräknar en **Grad-CAM-värmekarta** från sista conv-lagret, tröskar den
till en bounding box och jämför med YOLO-boxen via **IoU** (Intersection over Union).
Steget körs bara på positiva testbilder, eftersom det krävs en referensbox att
jämföra med.

Sammanfattning från den sparade rapporten (`output/grad_cam/grad_cam_iou_report.txt`):

| Mått | Värde |
|---|---|
| Positiva bilder | 486 |
| Genomsnittligt IoU | 0.15 |
| Korrekt lokaliserade (IoU ≥ 0.30) | 74 |
| Detektionsandel | 15.2 % |

Modellen är alltså bra på att *avgöra* om det finns en människa, men sämre på att
peka ut *exakt var*. Vid visuell inspektion ser vi också att den ibland aktiverar på
**bilar** — den har lärt sig "avvikande objekt i terräng" snarare än enbart människa.


In [ ]:
report_path = config.GRAD_CAM_OUTPUT_DIR / config.GRAD_CAM_REPORT_FILE_NAME
print("\n".join(report_path.read_text(encoding="utf-8").splitlines()[:12]))


Exempelbilder: original, Grad-CAM-overlay och bounding boxes (grön = ground truth,
röd = CAM-box).

![Grad-CAM exempel 1](output/grad_cam/grad_cam_sample_1.png)

![Grad-CAM exempel 3](output/grad_cam/grad_cam_sample_3.png)

![Grad-CAM exempel 5](output/grad_cam/grad_cam_sample_5.png)


## 9. Andra saker vi testat

Utöver baslinjen utforskade vi flera spår för att försöka pressa resultatet vidare.

### 9.1 Träning med koordinater (multi-task learning)
Datasetet innehåller bounding box-koordinater (`[x, y, w, h]`). Vi undersökte om de
kunde förbättra modellen genom att användas som "facit" under träningen. Det tvingade
modellen att lära sig bildinnehållet bättre och gav **93 % träffsäkerhet för Human** —
marginellt bättre än baslinjen.

Vi testade även en YOLO-liknande variant som skulle prediktera *var* människan fanns.
Modellen var mycket bra på att avgöra *om* det fanns en människa, men box-prediktionen
var bristfällig vid visuell inspektion. Därför förenklades den tillbaka till en binär
klassificerare, optimerad med **Focal Loss** och **tröskeljustering** för att balansera
hög Human Recall mot en rimlig nivå falsklarm.

**Resultat (Threshold Tuning):**
```
              precision    recall  f1-score   support
    no human       0.65      0.83      0.73        84
       human       0.97      0.92      0.95       486
    accuracy                           0.91       570
```

### 9.2 Slutsats kring lokalisering
Lokaliseringsträningen var det enskilt viktigaste steget för att tvinga modellen att
faktiskt lära sig se objektet "människa". Focal Loss + tröskeljustering gav den bästa
praktiska balansen. När vi jämför den utforskade modellen med den ursprungliga är
förbättringarna dock **marginella** — vilket tyder på att baslinjens arkitektur redan
var väl anpassad för uppgiften.

### 9.3 Utökat dataset med syntetisk data
Vi testade att lägga till syntetiska bilder (andra miljöer och vinklar, enbart
människor och inga bilar). Det gav **ingen förbättring** av resultatet.


## 10. Slutsats

Genom att balansera datasetet (class weights + augmentering) och välja en avvägning
som prioriterar **hög Human Recall** fick vi en modell som hittar de flesta människorna:

```
              precision    recall  f1-score   support
    no human       0.61      0.81      0.70        84
       human       0.97      0.91      0.94       486
    accuracy                           0.90       570
```

Grad-CAM visar att modellen tittar i ungefär rätt riktning men har svag exakt
lokalisering, och att den ibland förväxlar bilar med människor. För en skarp
tillämpning åt Försvarsmakten är hög recall det viktigaste — att hellre flagga för
mycket än att missa en nödställd person — men nästa steg vore att förbättra
lokaliseringen och minska förväxlingen med fordon.
